In [1]:
import pandas as pd
parent_dir = "data_phase1"
df_cell_info = pd.read_csv(f'{parent_dir}/cell_info.txt', sep='\t')     # Metadata for cell lines (the samples of the data matrices)
df_gene_info = pd.read_csv(f'{parent_dir}/gene_info.txt', sep='\t')     # Metadata for genes (the features of the data matrices)
df_inst_info = pd.read_csv(f'{parent_dir}/inst_info.txt', sep='\t')     # Metadata for L1000 experiments (the data matrices)
df_pert_info = pd.read_csv(f'{parent_dir}/pert_info.txt', sep='\t')     # Metadata for perturbations applied in L1000 experiments
df_sig_info = pd.read_csv(f'{parent_dir}/sig_info.txt', sep='\t')       # Metadata for level 5 profiles

/tmp/ipykernel_1088538/443091508.py:5: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  df_inst_info = pd.read_csv(f'{parent_dir}/inst_info.txt', sep='\t')     # Metadata for L1000 experiments (the data matrices)
/tmp/ipykernel_1088538/443091508.py:7: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.
  df_sig_info = pd.read_csv(f'{parent_dir}/sig_info.txt', sep='\t')       # Metadata for level 5 profiles


In [5]:
# load pw_in_cancer_gene_to_idx.pkl to get all gene names in the pathway
import pickle
with open('pw_in_cancer_gene_to_idx.pkl', 'rb') as f:
    gene_to_idx = pickle.load(f)
gene_names = list(gene_to_idx.keys())
print(f"Loaded {len(gene_names)} gene names from pw_in_cancer_gene_to_idx.pkl")

# sanity check: make sure all gene names are in df_gene_info
for gene in gene_names:
    assert gene in df_gene_info['pr_gene_symbol'].values, f"Gene {gene} not found in df_gene_info"

Loaded 12328 gene names from pw_in_cancer_gene_to_idx.pkl


In [2]:
# get value counts for pert_type in df_sig_info
pert_type_counts = df_sig_info['pert_type'].value_counts()
print("ℹ️ Perturbation types in sig_info:")
for pert_type, count in pert_type_counts.items():
    print(f" - {pert_type}: {count}")

ℹ️ Perturbation types in sig_info:
 - trt_cp: 205034
 - trt_sh: 154993
 - trt_sh.cgs: 36720
 - trt_sh.css: 24368
 - trt_oe: 22205
 - ctl_vehicle: 14423
 - trt_lig: 8256
 - ctl_vector: 6826
 - ctl_untrt: 588
 - ctl_vector.cns: 137
 - ctl_vehicle.cns: 61
 - ctl_untrt.cns: 30
 - trt_oe.mut: 6


In [ ]:

df_perturbed_genes = df_sig_info.merge(df_gene_info, left_on='pert_iname', right_on='pr_gene_symbol', how='inner')

In [ ]:
# filter df_perturbed_genes to only include rows where pr_gene_symbol is in the list of gene symbols in the graph G
print(f"ℹ️ Filtering perturbed genes from {len(df_perturbed_genes)} to only those present in the pathway graph.")
df_perturbed_genes = df_perturbed_genes[df_perturbed_genes['pr_gene_symbol'].isin(G.nodes)]
print(f"✅ Filtered perturbed genes to {len(df_perturbed_genes)} genes present in the pathway graph.")
# print pert types with their counts
pert_type_counts = df_perturbed_genes['pert_type'].value_counts()
print("ℹ️ Perturbation types in filtered perturbed genes:")
for pert_type, count in pert_type_counts.items():
    print(f" - {pert_type}: {count}")

In [ ]:
df_perturbed_genes
# get unique values for cell_id
cell_lines = df_perturbed_genes['cell_id'].unique().tolist()
print(f"Found {len(cell_lines)} unique cell lines with perturbations.")
#count for how many of those cell lines we have a row with pert_type 'ctl_untrt.cns'
control_cell_lines = df_sig_info[df_sig_info['pert_type'] == "ctl_untrt"]['cell_id'].nunique()
print(f"Found {control_cell_lines} unique cell lines with control (untreated) samples.")

# create a dataframe that has one row per cell line, with columns:
# cell_id, control_sample (sig_id of control sample), pert_type
# while creating the dataframe, ignore cell lines that do not have a control sample, but print the name of the cell line and how many rows it has in df_sig_info
# for the control sample, get the sig_id from df_sig_info where pert_type is 'ctl_untrt.cns' and cell_id matches. print how many rows were found for that cell line in df_sig_info
# if ctl_untrt.cns sample is not found, try ctl_untrt. if there are multiple ctl_untrt samples, pick the first one and print how many were found
control_pert_types = ['ctl_untrt.cns', 'ctl_untrt', 'ctl_vehicle.cns', 'ctl_vehicle']
control_samples = []
for cell_line in cell_lines:
    df_cell = df_sig_info[df_sig_info['cell_id'] == cell_line]
    for pert_type in control_pert_types:
        df_control = df_cell[df_cell['pert_type'] == pert_type]
        if not df_control.empty:
            break
    if not df_control.empty:
        control_sample = df_control.iloc[0]['sig_id']
        control_samples.append({'cell_id': cell_line, 'control_sample': control_sample})
    else:
        print(f"⚠️ No control sample found for cell line {cell_line} with {len(df_cell)} samples.")
df_control_samples = pd.DataFrame(control_samples)
print(f"✅ Created control samples dataframe with {len(df_control_samples)} entries.")
df_control_samples

In [ ]:
from pathlib import Path

# create a dataframe with three columns: unperturbed_sig_id, perturbed_sig_id, perturbed_gene

# Sanity checks (fail fast)
assert 'sig_id' in df_perturbed_genes.columns, "df_perturbed_genes must contain 'sig_id'"
assert 'pr_gene_symbol' in df_perturbed_genes.columns, "df_perturbed_genes must contain 'pr_gene_symbol'"
assert 'cell_id' in df_perturbed_genes.columns, "df_perturbed_genes must contain 'cell_id'"
assert 'cell_id' in df_control_samples.columns and 'control_sample' in df_control_samples.columns, \
    "df_control_samples must contain 'cell_id' and 'control_sample'"
assert Path(parent_dir).exists(), f"Parent directory not found: {parent_dir}"

df_perturbation_pairs = df_perturbed_genes.merge(df_control_samples, on='cell_id', how='inner')

if df_perturbation_pairs.empty:
    raise ValueError("No perturbation pairs found after merging df_perturbed_genes with df_control_samples")

df_perturbation_pairs = df_perturbation_pairs[['control_sample', 'sig_id', 'pr_gene_symbol']].rename(
    columns={
        'control_sample': 'unperturbed_sig_id',
        'sig_id': 'perturbed_sig_id',
        'pr_gene_symbol': 'perturbed_gene'
    }
)

out_path = Path(parent_dir) / 'perturbation_pairs.txt'
df_perturbation_pairs.to_csv(out_path, sep='\t', index=False)
print(f"✅ Wrote perturbation pairs with perturbed_gene to {out_path}")